In [11]:
from google.colab import userdata

key = userdata.get("OPENROUTER_API_KEY")
print("KEY FOUND:", key is not None)
print("KEY PREFIX:", repr(key[:12]) if key else None)

KEY FOUND: True
KEY PREFIX: 'sk-or-v1-278'


In [13]:
import requests
from tavily import TavilyClient
from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# Read secrets
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY").strip()
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY").strip()

# Model via OpenRouter
model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Tavily client
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

# Tool 1: internet search
@tool
def internet_search(query: str) -> str:
    """Search the internet and return the top 3 results."""
    result = tavily_client.search(
        query=query,
        max_results=3,
        topic="general",
        include_raw_content=False,
    )

    formatted = []
    for i, item in enumerate(result.get("results", []), start=1):
        formatted.append(
            f"Result {i}\n"
            f"Title: {item.get('title', '')}\n"
            f"URL: {item.get('url', '')}\n"
            f"Snippet: {item.get('content', '')}\n"
        )

    return "\n\n".join(formatted) if formatted else "No results found."

# Tool 2: fetch URL
@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL."""
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, timeout=10, headers=headers)
    response.raise_for_status()
    return response.text[:12000]

# Agent prompt
AGENT_PROMPT = """
You are a research assistant.

Your task:
1. Understand the user's question.
2. Optionally refine the search query if needed.
3. Use internet_search(query).
4. Select the top 3 most relevant URLs.
5. Use fetch_url(url) to read those pages.
6. Combine the information from those pages.
7. Write a clear final answer.
8. At the end, include a section called Sources with the URLs you used.
"""

# Create agent
agent = create_agent(
    model=model,
    tools=[internet_search, fetch_url],
    system_prompt=AGENT_PROMPT
)

# Ask a question
result = agent.invoke({
    "messages": [
        HumanMessage("What is the difference between LangChain and LangGraph?")
    ]
})

print(result["messages"][-1].content)

**LangChain vs. LangGraph – Key Differences**

| Aspect | LangChain | LangGraph |
|--------|-----------|-----------|
| **Core purpose** | A flexible, modular library for chaining together LLM calls, utilities, and other components to create LLM‑powered workflows. | A visual, low‑code extension of LangChain that focuses on building **stateful, adaptive agent systems** with explicit control over workflow state. |
| **Complexity handling** | Best suited for linear or moderately complex pipelines where rapid prototyping and a wide ecosystem of integrations are priorities. | Designed for complex, multi‑step reasoning loops, branching, and adaptive workflows that need explicit state management. |
| **State management** | Uses implicit state handling; the framework manages data flow behind the scenes. | Provides **explicit state control**, making it easier to track, modify, and persist data as the workflow evolves. |
| **User interface** | Primarily code‑first; developers write Python (or oth

In [12]:
from google.colab import userdata
from langchain_openai import ChatOpenAI

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY").strip()

model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

response = model.invoke("Say hello in exactly three words.")
print("RAW RESPONSE:", response)
print("CONTENT:", repr(response.content))

RAW RESPONSE: content='Hello dear friend' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 254, 'prompt_tokens': 23, 'total_tokens': 277, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 259, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-nano-30b-a3b:free', 'system_fingerprint': None, 'id': 'gen-1773175701-wJjWwbl0CFbi8j9sDWsJ', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cd981-fda7-78d2-ba50-4a6829e620dd-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 23, 'output_tokens': 254, 'total_tokens': 277, 'input_token_details':

In [10]:
import os
import requests
from tavily import TavilyClient

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage

from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY").strip()

model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL"""
    response = requests.get(url, timeout=10)
    return response.text[:10000]

AGENT_PROMPT = """
You are a research assistant.

Search the internet, read the top results, and answer clearly.
"""

agent = create_agent(
    model=model,
    tools=[fetch_url],
    system_prompt=AGENT_PROMPT
)

result = agent.invoke({
    "messages": [
        HumanMessage("What is LangGraph?")
    ]
})

print(result["messages"][-1].content)

**LangGraph** is a low‑level **agent orchestration framework** designed to help developers build **controllable, reliable AI agents**. It is part of the broader **LangChain ecosystem** and focuses on giving you fine‑grained control over how an agent’s components (nodes, edges, state, memory, etc.) are wired together and executed.

Key points from the official LangGraph page:

- **Agent orchestration** – It provides a graph‑based structure where each step of an agent’s workflow is represented as a node or edge, making it easy to compose complex, multi‑step behaviors.
- **Reliability & control** – By exposing the underlying graph, LangGraph lets you enforce deterministic execution paths, handle retries, and manage state transitions in a predictable way.
- **Integration with LangChain** – LangGraph works seamlessly with other LangChain building blocks (LLMs, memory, tools, etc.), allowing you to augment existing LangChain agents with richer orchestration capabilities.
- **Stateful & persi